# Hyperparameter tuning

Ottimizziamo il modello campione Random Forest e XGBoost (il modello con più margine di miglioramento)

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier

from src.data_loading import load_raw_data, build_working_dataset
from src.features import build_all_features
from src.train import temporal_split, FEATURE_COL

In [2]:
data = load_raw_data()
df = build_working_dataset(data["races"], data["results"], 2004)
df = build_all_features(df, 10)
train_df, test_df = temporal_split(df, 2023)

X_train, y_train = train_df[FEATURE_COL], train_df["podium"]
X_test, y_test = test_df[FEATURE_COL], test_df["podium"]

# Definiamo 5 fold temporali sul train set: il tuning avviene solo su questi dati
tscv = TimeSeriesSplit(n_splits=5)

## Tuning Random Forest

Esploriamo lo spazio intorno alla configurazione attuale

In [4]:
rf_param_grid = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [4, 6, 8, 10, 12],
    "min_samples_split": [2, 5, 8, 10],     # Numero minimo di righe per dividere un nodo -> valori più alti, alberi meno "sfrangiati", riduce overfitting
    "min_samples_leaf": [1, 2, 3, 4],       # Numero minimo di righe in una foglia finale 
    "max_features": ["sqrt", "log2", None]  # Quante feature considerare ad ogni split
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1),
    param_distributions=rf_param_grid,
    n_iter=30,                      # 30 combinazioni casuali invce di testarle tutte
    scoring="average_precision",    # area sotto la curva precision-recall
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Migliori parametri Random Forest:")
print(rf_search.best_params_)
print(f"\nMiglior average_precision in CV: {rf_search.best_score_:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Migliori parametri Random Forest:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 4}

Miglior average_precision in CV: 0.6558


## Tuning XGBoost

XGBoost ha molti più iperparametri rilevanti di Random Forest, in particolare quelli legati alla regolarizzazione e al learning rate 


In [5]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_param_grid = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2], # quanto "aggressivamente" ogni nuovo albero corregge gli errori dei precedenti: valori bassi = training più lento mapiù preciso, con più alberi
    "subsample": [0.6, 0.8, 1.0],            # frazione di righe usate per ogni albero: <1.0 introduce casualità, riduce overfitting
    "colsample_bytree": [0.6, 0.8, 1.0],     # frazione di feature usate per ogni albero
    "reg_alpha": [0, 0.1, 1],                # regolarizzazione L1 (penalizza pesi grandi)
    "reg_lambda": [1, 1.5, 2],               # regolarizzazione L2
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss"
    ),
    param_distributions=xgb_param_grid,
    n_iter=30,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

print("Migliori parametri XGBoost:")
print(xgb_search.best_params_)
print(f"\nMiglior average_precision in CV: {xgb_search.best_score_:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Migliori parametri XGBoost:
{'subsample': 0.6, 'reg_lambda': 2, 'reg_alpha': 1, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.8}

Miglior average_precision in CV: 0.6666


## Valutazione finale sul test set

Ora portiamo i due modelli ottimizzati sul test set 2023-2024 (mai visto durante il tuning), cercando per ciascuno la soglia di decisione ottimale

In [6]:
def find_best_threshold(model, X_test, y_test, thresholds=np.arange(0.1, 0.95, 0.05)):
    y_proba = model.predict_proba(X_test)[:,1]
    best_f1, best = -1, None
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_test, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best = {"threshold": t,
                    "precision": precision_score(y_test, y_pred),
                    "recall": recall_score(y_test, y_pred),
                    "f1": f1}

    return best

rf_tuned_result = find_best_threshold(rf_search.best_estimator_, X_test, y_test)
xgb_tuned_result = find_best_threshold(xgb_search.best_estimator_, X_test, y_test)

comparison = pd.DataFrame([
    {"model": "Random Forest (originale)", "threshold": 0.60, "precision": 0.545, "recall": 0.884, "f1": 0.674},
    {"model": "Random Forest (tuned)", **rf_tuned_result},
    {"model": "XGBoost (originale)", "threshold": 0.40, "precision": 0.509, "recall": 0.855, "f1": 0.638},
    {"model": "XGBoost (tuned)", **xgb_tuned_result},
]).sort_values("f1", ascending=False)

print(comparison.to_string(index=False))

                    model  threshold  precision   recall       f1
          XGBoost (tuned)       0.75   0.628049 0.746377 0.682119
Random Forest (originale)       0.60   0.545000 0.884000 0.674000
    Random Forest (tuned)       0.80   0.664286 0.673913 0.669065
      XGBoost (originale)       0.40   0.509000 0.855000 0.638000


## Risultati

Il tuning non garantisce sempre un miglioramento sulla metrica che conta davvero, dipende da cosa ottimizzi.
Il tuning ha ribaltato il risultato del confronto iniziale: **XGBoost, opportunamente ottimizzato, supera Random Forest** (F1 0.682 vs 0.674), principalmente grazie a una precision molto più alta (0.628 vs 0.545).
**Prossimo passo**: costruire la pipeline di aggiornamento dati (Jolpica-F1), che userà questo modello per generare previsioni su gare reali non ancora disputate.

In [9]:
importances_xgb = pd.DataFrame({
    "feature": FEATURE_COL,
    "importance": xgb_search.best_estimator_.feature_importances_
}).sort_values("importance", ascending=False)

print(importances_xgb)

                       feature  importance
0                         grid    0.490355
2   driver_recent_position_avg    0.319446
1     driver_recent_points_avg    0.091568
5           no_circuit_history    0.041241
4  driver_circuit_avg_position    0.034391
3      constructor_reliability    0.022998


In [7]:
import sys
sys.path.append("..")
from src.train import save_model

# XGBoost tunato è ora il modello campione, sostituendo Random Forest
save_model(xgb_search.best_estimator_, threshold=0.75, output_dir="../models")

Modello salvato in ../models/random_forest_final.pkl
Configurazione salvata in ../models/model_config.json
